# Vytvoření znovupoužitelné aktuárské cenotvorné knihovny pomocí PROC FCMP

## Shrnutí pro vedení

Pojišťovna majetkového a úrazového pojištění zapouzdřuje svou základní cenotvornou matematiku — čisté pojistné, přirážku za náklady/riziko, kredibilní směšování s omezenou fluktuací a odhad diskontované rezervy — jako vlastní funkce a víceodstupovou subrutinu v **PROC FCMP**. Kompilovaná knihovna je zaregistrována pomocí systémové volby `CMPLIB=` a poté volána řádek po řádku z kroku DATA, který oceňuje syntetické portfolio 100 pojistek. Každá oceněná hodnota v tomto notebooku — kredibilní faktor `z`, smíšené čisté pojistné, účtované pojistné a diskontovaná rezerva na škodu — je vytvořena těmito kompilovanými rutinami, nikoli vloženou aritmetikou. Výsledek: implikovaný škodní poměr dosahuje 60,5 % (venkovský region), 55,8 % (příměstský region) a 63,6 % (městský region) — pohodlně pod 100 % v každém segmentu, což potvrzuje, že navýšené pojistné pokrývá očekávanou škodu, zatímco krok ocenění zůstává čistý a auditovatelný.

## Zdroje dat

| Datová sada | Řádky | Popis | Klíčové proměnné |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Syntetické aktivní portfolio pojistek majetkového a úrazového pojištění generované přímo pomocí `rand()` | `policy_id`, `region` (městská/příměstská/venkovská), `years_insured`, `exposure` (roky vozidla), `claim_count` (Poissonovo rozdělení), `incurred_loss` (gama závažnost x počet), `class_pure_prem` (manuální sazba podle regionu) |

Krok DATA prochází širším rozsahem `policy_id`, ale toto prostředí běží v nelicencovaném režimu, takže výstup je omezen na prvních **100 pozorování** — oceněná kniha níže odráží těchto 100 pojistek.

# Vytvoření znovupoužitelné aktuárské cenotvorné knihovny pomocí PROC FCMP

Aktuáři opakují stejné výpočty napříč úlohami cenotvorby, tvorby rezerv a reportingu: převod škod na *čisté pojistné*, aplikaci *přirážek za náklady a riziko* pro dosažení účtovaného pojistného, směšování vlastní zkušenosti jednotlivé pojistky s třídní sazbou pomocí *kredibility* a *diskontování* budoucích peněžních toků na současnou hodnotu. Opětovné psaní těchto vzorců v každém kroku DATA vede k chybám při kopírování a znesnadňuje změny předpokladů.

**PROC FCMP** (kompilátor funkcí SAS) nám umožňuje definovat každý vzorec jen jednou jako pojmenovanou funkci nebo subrutinu, uložit kompilované rutiny do knihovny a poté je volat jako jakoukoli vestavěnou funkci SAS. V tomto notebooku:

1. Zkompilujeme malou knihovnu aktuárských funkcí pomocí `PROC FCMP`.
2. Zaregistrujeme ji pro relaci pomocí systémové volby `CMPLIB=`.
3. Vygenerujeme syntetické portfolio majetkového a úrazového pojištění.
4. Oceníme každou pojistku voláním našich vlastních funkcí a víceodstupové subrutiny z jediného kroku DATA.
5. Shrneme a interpretujeme oceněné portfolio.

## Krok 1 — Generování syntetického portfolia pojistek

Simulujeme knihu aktivních autopojistek (prvních 100 je oceněno níže v rámci limitu nelicencovaného režimu). Každá pojistka patří do sazebního regionu s vlastním manuálním *čistým pojistným* (očekávaná škoda na rok vozidla). Počty škod se řídí Poissonovým procesem škálovaným expozicí a závažnosti se řídí gama rozdělením, takže `incurred_loss` je realistická skládaná (Poisson x gama) škoda. `years_insured` bude později určovat váhu kredibility. Pevné počáteční semínko přes `call streaminit` činí data reprodukovatelnými.

In [1]:
data portfolio;
    CALL streaminit(20260531);
    DÉLKA region $16;
    OPAKUJ policy_id = 10001 TO 12000;
        /* Přiřazení sazebního regionu */
        u = rand('uniform');
        KDYŽ u < 0.40 PAK OPAKUJ; region = 'MĚSTSKÁ';    base_pp = 820; lambda = 0.18; KONEC;
        JINAK KDYŽ u < 0.75 PAK OPAKUJ; region = 'PŘÍMĚSTSKÁ'; base_pp = 540; lambda = 0.11; KONEC;
        JINAK OPAKUJ; region = 'VENKOVSKÁ';    base_pp = 360; lambda = 0.07; KONEC;

        /* Doba pojištění (roky) a expozice (roky vozidla) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Skládaný proces škod: Poissonova četnost, gama závažnost */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        OPAKUJ c = 1 TO claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        KONEC;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manuální třídní čisté pojistné pro expozici této pojistky */
        class_pure_prem = round(base_pp * exposure, 0.01);

        VÝSTUP;
    KONEC;
    PONECHAT policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
SPUSTIT;

PROCEDURA TISK data=portfolio(obs=8) noobs;
    NÁZEV 'Prvních 8 simulovaných pojistek';
SPUSTIT;

                                            Prvních 8 simulovaných pojistek                                             

policy_id          region  years_insured  exposure  claim_count  incurred_loss  class_pure_prem
    10001  VENKOVSKÁ                   5      1.36            0              0            489.6
    10002  MĚSTSKÁ                     8      0.97            1        3432.28            795.4
    10003  MĚSTSKÁ                     2      1.53            2        7155.92           1254.6
    10004  PŘÍMĚSTSKÁ                  9       2.4            0              0             1296
    10005  VENKOVSKÁ                   5      0.99            0              0            356.4
    10006  PŘÍMĚSTSKÁ                  1      0.83            0              0            448.2
    10007  VENKOVSKÁ                   5      0.97            0              0            349.2
    10008  VENKOVSKÁ                   7      2.32            0              0            835.2

... 92 more o


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.43 seconds
  cpu   0.43 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Krok 2 — Kompilace knihovny aktuárských funkcí

Nyní srdce notebooku. `PROC FCMP` s `OUTLIB=work.actfuncs.pricing` kompiluje čtyři funkce a jednu subrutinu do balíčku `pricing` datové sady `work.actfuncs`:

- **`pure_premium`** — pozorovaná škoda na jednotku expozice (kombinace četnosti x závažnosti).
- **`credibility_z`** — kredibilní faktor s omezenou fluktuací `Z = sqrt(n / (n + k))`, kde `n` je počet let expozice pojistky a `k` je ladicí konstanta.
- **`charged_premium`** — aplikuje proporcionální rizikovou přirážku a pevný poměr nákladů na náklad na škody, aby se dosáhlo pojistného, které skutečně účtujeme.
- **`pv_reserve`** — současná hodnota budoucí výplaty škody, `FV / (1+r)**t`, používaná k diskontování rezerv na škody.
- **`blend_premium`** (SUBRUTINA) — používá `OUTARGS` k vrácení *dvou* hodnot najednou: kredibilně vážené čisté pojistné a použitý kredibilní faktor, takže krok DATA získá obě hodnoty v jediném volání.

Každá rutina je uzavřena pomocí `ENDSUB` a subrutina pojmenovává své zapisovatelné argumenty pomocí `OUTARGS`.

In [2]:
PROCEDURA fcmp outlib=work.actfuncs.pricing;

    /* Pozorované čisté pojistné: náklad na škody na jednotku expozice */
    function pure_premium(loss, exposure);
        KDYŽ exposure <= 0 PAK RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Kredibilita s omezenou fluktuací: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        KDYŽ n_years <= 0 PAK RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Účtované pojistné = náklad na škody navýšený o rizikovou přirážku a náklady */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        KDYŽ expense_ratio >= 1 PAK RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Současná hodnota budoucí výplaty škody diskontované t let sazbou r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Kredibilní směs: vrací vážené čisté pojistné A použité Z */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

SPUSTIT;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Krok 3 — Registrace knihovny pomocí CMPLIB=

Kompilace rutin nestačí; SAS musí být informován, kde je má hledat, když pozdější krok DATA nebo procedura odkazuje na jméno, které nerozpozná jako vestavěné. Systémová volba `CMPLIB=` ukazuje na datovou sadu (nikoli na třístupňový název balíčku) obsahující kompilovaný kód. Po tomto příkazu `OPTIONS` je každá výše uvedená funkce a subrutina volatelná podle jména.

In [3]:
VOLBY cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Krok 4 — Ocenění portfolia pomocí vlastních funkcí

Krok DATA pro ocenění je nyní téměř bez aritmetiky — aktuárský záměr je čitelný přímo z názvů funkcí. Pro každou pojistku:

1. Vypočítáme vlastní pozorované čisté pojistné pojistky pomocí `pure_premium`.
2. Zavoláme subrutinu `blend_premium` pro kredibilní vážení vůči regionální třídní sazbě a získáme jak smíšený náklad na škody, tak kredibilní faktor `z` prostřednictvím `OUTARGS`.
3. Navýšíme smíšený náklad na škody na účtované pojistné s 12% rizikovou přirážkou a 25% poměrem nákladů pomocí `charged_premium`.
4. Odhadneme stále otevřenou rezervu na škodu jako 35 % vzniklé škody pojistky a diskontujeme ji na tři roky při 4 % na současnou hodnotu pomocí `pv_reserve`.

Všimněte si, že výstupní argumenty subrutiny (`blended_pp`, `z`) musí být inicializovány před `CALL`. PV rezervy se liší pojistku od pojistky, protože je řízena skutečnou vzniklou škodou každé pojistky — pojistky bez škody mají nulovou rezervu, takže jejich `reserve_pv` je také nula.

In [4]:
data rated;
    NASTAVIT portfolio;

    /* 1. Vlastní škodní zkušenost pojistky jako čisté pojistné */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Kredibilní vážení vlastní zkušenosti vůči třídní sazbě.
          k = 4 roky expozice pro téměř úplnou kredibilitu. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Převod smíšeného nákladu na škody (na rok vozidla) na účtované pojistné */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Nevyřízená rezerva na škodu = 35 % vzniklé škody, očekávané
          vyrovnání za 3 roky; diskontováno na současnou hodnotu při 4 %. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    PONECHAT policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
SPUSTIT;

PROCEDURA TISK data=rated(obs=10) noobs;
    NÁZEV 'Oceněné portfolio (prvních 10 pojistek)';
    PROMĚNNÁ policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
SPUSTIT;

                                        Oceněné portfolio (prvních 10 pojistek)                                         

policy_id          region  years_insured  exposure   own_pp  blended_pp      z  premium  reserve_pv
    10001  VENKOVSKÁ                   5      1.36        0       91.67  0.745   186.18           0
    10002  MĚSTSKÁ                     8      0.97  3538.43     3039.59  0.816  4402.95     1067.95
    10003  MĚSTSKÁ                     2      1.53  4677.07     3046.88  0.577  6961.51     2226.55
    10004  PŘÍMĚSTSKÁ                  9       2.4        0       90.69  0.832   325.03           0
    10005  VENKOVSKÁ                   5      0.99        0       91.67  0.745   135.52           0
    10006  PŘÍMĚSTSKÁ                  1      0.83        0       298.5  0.447   369.98           0
    10007  VENKOVSKÁ                   5      0.97        0       91.67  0.745   132.79           0
    10008  VENKOVSKÁ                   7      2.32        0       72.82  0.798


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Krok 5 — Shrnutí oceněné knihy

Protože je každá pojistka oceněna prostřednictvím stejné kompilované knihovny, můžeme portfolio agregovat podle regionu. Reportujeme průměrné účtované pojistné, průměrný kredibilní faktor, celkovou vzniklou škodu a celkovou diskontovanou rezervu na škody, poté odvodíme implikovaný škodní poměr pro každý segment, abychom viděli, zda navýšené pojistné pokrývá očekávanou škodu.

In [5]:
PROCEDURA PRŮMĚRY data=rated mean sum maxdec=2 nonobs;
    TŘÍDA region;
    PROMĚNNÁ premium z incurred_loss reserve_pv;
    ŠTÍTEK region='Region'
          premium='Účtované pojistné'
          z='Kredibilita (z)'
          incurred_loss='Vzniklá škoda'
          reserve_pv='Současná hodnota rezervy';
    NÁZEV 'Souhrn portfolia podle sazebního regionu';
SPUSTIT;

/* Implikovaný škodní poměr = vzniklá škoda / účtované pojistné, plus
   diskontovaná rezerva, kterou region drží, podle regionu. */
PROCEDURA SQL;
    NÁZEV 'Implikovaný škodní poměr a PV rezervy podle regionu';
    VYBRAT region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     FORMÁT=dollar12.2,
           sum(premium)                          AS total_premium  FORMÁT=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     FORMÁT=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve FORMÁT=dollar12.2
    FROM rated
    GROUP PODLE region
    ORDER PODLE region;
QUIT;
NÁZEV;

                                        Souhrn portfolia podle sazebního regionu                                        

                                                  The MEANS Procedure

                                   Analysis Variable : premium Účtované pojistné

        Region                   Mean            Sum
        --------------------------------------------
        MĚSTSKÁ               1987.17       67563.93
        PŘÍMĚSTSKÁ             813.04       34147.74
        VENKOVSKÁ              476.61       11438.62
        --------------------------------------------

                                         Analysis Variable : z Kredibilita (z)

        Region                   Mean            Sum
        --------------------------------------------
        MĚSTSKÁ                  0.70          23.90
        PŘÍMĚSTSKÁ               0.68          28.36
        VENKOVSKÁ                0.71          17.14
        --------------------------------------------

       


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpretace výsledků

Krok DATA pro ocenění nikde přímo nevypisuje jediný diskontní nebo kredibilní vzorec inline — pouze volá `pure_premium`, `blend_premium`, `charged_premium` a `pv_reserve`. To je přínos **PROC FCMP**: aktuárské předpoklady žijí v jedné kompilované knihovně, kterou lze jednotkově testovat, verzovat a znovu použít napříč úlohami cenotvorby, tvorby rezerv a reportingu. Změna kredibilní konstanty `k`, rizikové přirážky nebo poměru nákladů je jedinou úpravou v knihovně, nikoli pátráním v každém programu.

Při čtení oceněné knihy a regionálního souhrnu:

- **Kredibilita (`z`)** roste s `years_insured`, přesně jak předepisuje vzorec s omezenou fluktuací `Z = sqrt(n / (n + k))`. Mezi prvními deseti pojistkami dosahuje jednoletá příměstská pojistka (10006) hodnoty `z = 0,447`, dvouletá městská pojistka (10003) `z = 0,577`, zatímco devítiletá příměstská pojistka (10004) dosahuje `z = 0,832`. Pojistky s krátkou zkušeností jsou taženy k regionální třídní sazbě; dlouhodobě pojištěné se opírají o vlastní škody.
- **Směšování v akci:** pojistky bez škody (většina knihy) mají `own_pp = 0`, takže `blend_premium` vrací `blended_pp` blízké `(1 - z)` násobku třídní sazby — např. pojistka 10001 (venkovský region, `z = 0,745`) dosahuje `91,67` proti třídní sazbě `360`/rok vozidla. Dvě městské pojistky, které měly škodu, 10002 a 10003, naopak táhnou svůj smíšený náklad na škody směrem k vlastní vysoké zkušenosti (`3039,59` a `3046,88`).
- **Účtované pojistné** je vyšší než smíšený náklad na škody, protože `charged_premium` přidává 12% rizikovou přirážku a navyšuje o 25% poměr nákladů, což je pevný násobitel `1,12 / 0,75 = 1,493` na náklad na škody. Průměrné účtované pojistné činí `476,61` (venkovský region), `813,04` (příměstský region) a `1 987,17` (městský region).
- **Diskontované rezervy:** `pv_reserve` diskontuje nevyřízenou rezervu na škodu každé pojistky (35 % vzniklé škody) na tři roky při 4 %, což je faktor současné hodnoty `0,889`. Pojistky bez škody mají `reserve_pv = 0`; dvě městské pojistky se škodou přispívají `1 067,95` a `2 226,55`. V souhrnu drží kniha diskontovanou rezervu `2 151,56 $` (venkovský region), `5 932,52 $` (příměstský region) a `13 364,13 $` (městský region).
- **Implikované škodní poměry** dosahují 60,5 % (venkovský region), 55,8 % (příměstský region) a 63,6 % (městský region) — všechny pohodlně pod 100 %, takže navýšené pojistné pokrývá očekávanou škodu v každém segmentu. Městský segment běží nejžhavěji, tažen svými dvěma velkými simulovanými škodami; skutečná revize sazeb by otestovala, zda tento signál přetrvává napříč více pojistnými roky, než by se upravila manuální sazba.

Subrutina `blend_premium` také demonstruje idiom `OUTARGS` pro vrácení více výsledků z jednoho `CALL` — smíšený náklad na škody a kredibilní faktor `z` se vrací společně — čímž se vyhýbá samostatným voláním funkcí a udržuje logiku ocenění na úrovni jednotlivé pojistky kompaktní a auditovatelnou.